## Dim table cleaning and ingesting into Silver ##

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
catalog_name = "ecommerce"

## Cleaning brands table

In [0]:
df_bronze_brands = spark.read.table(f"{catalog_name}.bronze.brz_brands")
display(df_bronze_brands.limit(10))


In [0]:
#Trimming leading/trialing spaces in the colum brand_name
df_silver_brands = df_bronze_brands.withColumn("brand_name", F.trim(F.col("brand_name")))
df_silver_brands = df_silver_brands.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), "[^a-zA-Z0-9]", ""))
anamolies = {
  "GROCERY" : "GRCY",
  "TOYS" : "TOY",
  "BOOKS" : "BKS"
}
df_silver_brands = df_silver_brands.replace(to_replace=anamolies, subset="category_code")

display(df_silver.limit(10))
#Writing the silver table
df_silver_brands.write.format("delta").mode("overwrite").option("mergeschema", True).saveAsTable(f"{catalog_name}.silver.slv_brands")

## Cleaning Category Table ##

In [0]:
df_bronze_category = spark.read.table(f"{catalog_name}.bronze.brz_category")
df_duplicates = df_bronze_category.groupBy("category_code").count().filter(F.col("count") > 1)
display(df_duplicates)
df_silver_category = df_bronze_category.dropDuplicates(["category_code"])
df_silver_category = df_silver_category.withColumn("category_code", F.upper(F.col("category_code")))
df_silver_category.show()
#Writing the silver table
df_silver_category.write.format("delta").mode("overwrite").option("mergeschema", True).saveAsTable(f"{catalog_name}.silver.slv_category")


## CLeaning Customers Table ##

In [0]:
from pyspark.sql.functions import col
df_bronze_customers = spark.read.table(f"{catalog_name}.bronze.brz_customers")

display(df_bronze_customers.limit(10))
df_silver_customers = df_bronze_customers.dropna(subset=["customer_id"])
null_count = df_silver_customers.filter(col("phone").isNull()).count()
print(null_count)
df_silver_customers = df_silver_customers.fillna("Not Available", subset="phone")
## Writing the silver table
df_silver_customers.write.format("delta").mode("overwrite").option("mergeschema", True).saveAsTable(f"{catalog_name}.silver.slv_customers")





## Cleaning products table ##

In [0]:
df_bronze_products = spark.read.table(f"{catalog_name}.bronze.brz_products")
display(df_bronze_products.limit(10))
df_bronze_products.filter(F.col("brand_code").isNull()).show()
df_silver_products = df_bronze_products.withColumn("brand_code", F.upper(F.col("brand_code"))).withColumn("category_code", F.upper(F.col("category_code")))
df_silver_products = df_silver_products.withColumn("weight_grams", F.regexp_replace(F.col("weight_grams"), "g", "").cast("int"))
df_silver_products = df_silver_products.withColumn("length_cm", F.regexp_replace(F.col("length_cm"),",",".").cast("float"))
df_silver_products = df_silver_products.withColumn("rating_count", F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count"))).otherwise(0))


df_silver_products.select("material").distinct().show()

df_silver_products = df_silver_products.withColumn("material", F.when(F.col("material") == "Coton", "Cotton").when(F.col("material") == "Ruber", "Rubber").when(F.col("material") == "Alumium", "Aluminium").otherwise(F.col("material")))
display(df_silver_products.limit(10))
df_silver_products.select("material").distinct().show()
#writing the silver table
df_silver_products.write.format("delta").mode("overwrite").option("mergeschema", True).saveAsTable(f"{catalog_name}.silver.slv_products")
                                            



## Cleaning date table ##


In [0]:
df_bronze_date = spark.read.table(f"{catalog_name}.bronze.brz_date")
display(df_bronze_date.limit(10))
df_duplicates = df_silver_date.groupBy("date").count().filter(F.col("count") > 1)
print(df_duplicates.count())
display(df_duplicates)
df_silver_date = df_bronze_date.dropDuplicates(["date"])
df_silver_date = df_silver_date.withColumn("day_name", F.initcap(F.col("day_name")))
df_silver_date = df_silver_date.withColumn("week_of_year",F.abs(F.col("week_of_year"))).withColumn("week_of_year", F.concat(F.lit("WEEK"), F.col("week_of_year"),F.lit(" - "),F.col("year")))
df_silver_date = df_silver_date.withColumn("quarter",F.abs(F.col("quarter"))).withColumn("quarter", F.concat(F.lit("Q"), F.col("quarter"),F.lit(" - "),F.col("year")))
df_silver_date = df_silver_date.drop("month")
display(df_silver_date.limit(10))
#writing the silver table
df_silver_date.write.format("delta").mode("overwrite").option("mergeschema", True).saveAsTable(f"{catalog_name}.silver.slv_date")
